# 🧪 SABER Advanced Features & Post-Processing Evaluation Notebook

This notebook evaluates the **SABER** framework with all game-changing accuracy upgrades:
- **Multi-Layer ViT Feature Fusion** (Blocks 6+9+12)
- **PCA Whitening Transform** (768-D → 512-D)
- **Database-Side Feature Augmentation (DBA)** (Gallery 5-NN Smoothing)
- **Alpha Query Expansion (αQE)** (Top-3 Weighted Query Expansion)
- **Mean-Centering & Cosine Calibration**

---

## 📁 Step 1: Mount Google Drive
Mount Google Drive to fetch your trained checkpoints (`saber_unified.pth`, `bridge_unified.pth`) and dataset archives.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 🛠️ Step 2: Clone Repository & Install Dependencies

In [ ]:
import os

# Clone or pull latest SABER code
if os.path.exists("/content/SABER"):
    print("SABER directory exists. Pulling latest code...")
    %cd /content/SABER
    !git pull
else:
    print("Cloning SABER repository...")
    !git clone https://github.com/SK8-infi/SABER.git
    %cd SABER

# Prevent PEFT/torchao conflicts on Colab
!pip uninstall -y torchao

# Install requirements
!pip install -r Saber/requirements.txt
!pip install albumentations --upgrade

# Cache DOFA base weights
!mkdir -p /root/.cache/torch/hub/checkpoints/
drive_weights = "/content/drive/MyDrive/SABER_Data/DOFA_ViT_base_e100.pth"
local_weights = "/root/.cache/torch/hub/checkpoints/DOFA_ViT_base_e100.pth"

if os.path.exists(drive_weights):
    print("Copying DOFA weights from Google Drive...")
    !cp "{drive_weights}" "{local_weights}"
else:
    print("Downloading DOFA weights from HuggingFace...")
    !huggingface-cli download earthflow/DOFA DOFA_ViT_base_e100.pth --local-dir /root/.cache/torch/hub/checkpoints/ --local-dir-use-symlinks False

## 📥 Step 3: Fetch Trained Checkpoints from Google Drive

In [ ]:
import os

!mkdir -p checkpoints/
drive_ckpt_dir = "/content/drive/MyDrive/SABER_Data/checkpoints"

ckpts_to_fetch = ["saber_unified.pth", "bridge_unified.pth", "bridge_best.pth", "latest.pth"]
fetched_count = 0

for ckpt in ckpts_to_fetch:
    drive_path = os.path.join(drive_ckpt_dir, ckpt)
    local_path = os.path.join("checkpoints", ckpt)
    if os.path.exists(drive_path):
        print(f"Fetching '{ckpt}' from Google Drive...")
        !cp "{drive_path}" "{local_path}"
        fetched_count += 1

if fetched_count == 0:
    print("⚠️ No checkpoints found in Google Drive! Check if path is '/content/drive/MyDrive/SABER_Data/checkpoints/'")
else:
    print(f"Successfully fetched {fetched_count} checkpoint(s).")
    !ls -lh checkpoints/

## 📦 Step 4: Extract or Download Datasets

In [ ]:
import os

!mkdir -p Datasets/DSRSID
!mkdir -p Datasets/benv1_14k

drive_saber_data = "/content/drive/MyDrive/SABER_Data"

# 1. DSRSID Setup
dsrsid_zip = os.path.join(drive_saber_data, "DSRSID-001.zip")
dsrsid_zip_alt = os.path.join(drive_saber_data, "DSRSID.zip")

if os.path.exists(dsrsid_zip):
    print("Unzipping DSRSID-001.zip...")
    !unzip -q "{dsrsid_zip}" -d Datasets/DSRSID/
elif os.path.exists(dsrsid_zip_alt):
    print("Unzipping DSRSID.zip...")
    !unzip -q "{dsrsid_zip_alt}" -d Datasets/DSRSID/
else:
    raw_mat = os.path.join(drive_saber_data, "DSRSID/DSRSID-001.mat")
    if os.path.exists(raw_mat):
        !ln -s "{raw_mat}" Datasets/DSRSID/DSRSID-001.mat

# Clean up nested folder structure if present
if os.path.exists("Datasets/DSRSID/DSRSID/DSRSID-001.mat"):
    !mv Datasets/DSRSID/DSRSID/* Datasets/DSRSID/ 2>/dev/null || true
    !rmdir Datasets/DSRSID/DSRSID 2>/dev/null || true

# 2. BEN-14K Setup
ben_tgz = os.path.join(drive_saber_data, "benv1_14k.tgz")
ben_zip = os.path.join(drive_saber_data, "benv1_14k.zip")

if os.path.exists(ben_tgz):
    print("Extracting benv1_14k.tgz...")
    !tar -xf "{ben_tgz}" -C Datasets/
elif os.path.exists(ben_zip):
    print("Unzipping benv1_14k.zip...")
    !unzip -q "{ben_zip}" -d Datasets/
else:
    raw_ben = os.path.join(drive_saber_data, "benv1_14k")
    if os.path.exists(raw_ben):
        !ln -s "{raw_ben}" Datasets/benv1_14k

if os.path.exists("Datasets/benv1_14k/benv1_14k"):
    !mv Datasets/benv1_14k/benv1_14k/* Datasets/benv1_14k/
    !rmdir Datasets/benv1_14k/benv1_14k 2>/dev/null || true

print("\n--- Verification of Datasets ---")
!ls -lh Datasets/DSRSID/
!ls -lh Datasets/benv1_14k/

## 🧪 Step 5: Evaluate BEN-14K with Post-Processing Pipeline (PCA Whitening + DBA + αQE)

In [ ]:
# ==========================================================
# BEN-14K EVALUATION (HELD-OUT TEST SPLIT)
# Uses PCA Whitening (768d->512d) + DBA (5-NN) + αQE (Top-3)
# ==========================================================

print("1. BEN-14K Same-Modal Retrieval (Sentinel-2 -> Sentinel-2)...")
!python Saber/evaluate.py --architecture saber --dataset_name ben14k --modality s2 --synthetic false --data_dir Datasets/benv1_14k --checkpoint checkpoints/saber_unified.pth --split test

print("\n2. BEN-14K Cross-Modal Retrieval (Sentinel-1 SAR -> Sentinel-2 Optical)...")
!python Saber/evaluate.py --architecture saber --dataset_name ben14k --modality both --synthetic false --data_dir Datasets/benv1_14k --checkpoint checkpoints/saber_unified.pth --split test --direction s1_to_s2

## 🧪 Step 6: Evaluate DSRSID with Post-Processing Pipeline (PCA Whitening + DBA + αQE)

In [ ]:
# ==========================================================
# DSRSID EVALUATION (PROPERLY SHUFFLED TEST SPLIT - SEED 42)
# Uses PCA Whitening (768d->512d) + DBA (5-NN) + αQE (Top-3)
# ==========================================================

print("1. DSRSID Same-Modal Retrieval (Multi-Spectral -> Multi-Spectral)...")
!python Saber/evaluate.py --architecture saber --dataset_name dsrsid --modality ms --synthetic false --data_dir Datasets/DSRSID --checkpoint checkpoints/saber_unified.pth --split test --size 14000

print("\n2. DSRSID Cross-Modal Retrieval (Panchromatic -> Multi-Spectral)...")
!python Saber/evaluate.py --architecture saber --dataset_name dsrsid --modality both --synthetic false --data_dir Datasets/DSRSID --checkpoint checkpoints/saber_unified.pth --split test --size 14000

## 📊 Step 7: Run Advanced EDA Diagnostics

In [ ]:
# Run EDA diagnostics to inspect per-class breakdowns, dynamic range gap, and label co-occurrences
!python Saber/eda_diagnostics.py --checkpoint checkpoints/saber_unified.pth